# Solar Filament Segmentation 2026 — U-Net + Connected Components

Simple, fast pipeline optimised for **Panoptic Quality**:

1. **U-Net** (`segmentation_models_pytorch`, pretrained ResNet-34, 1-channel input) predicts a filament probability map.
2. **Instances** come from post-processing: threshold → group nearby fragments (dilate-then-label) → drop tiny blobs.
3. Post-processing params are **grid-searched on validation PQ** (`torchmetrics.detection.PanopticQuality`).
4. Flip TTA at inference → `submission.csv` (pycocotools RLE).

In [ ]:
!pip install -q segmentation-models-pytorch albumentations pycocotools "torchmetrics>=1.4" scikit-image

In [ ]:
import json, itertools, random
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import albumentations as A
import segmentation_models_pytorch as smp
from albumentations.pytorch import ToTensorV2
from pycocotools import mask as mask_utils
from scipy import ndimage
from skimage.measure import regionprops
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection import PanopticQuality
from tqdm.auto import tqdm

CFG = dict(
    img_size=1024,          # 2048 originals are downscaled 2x for training
    encoder="resnet34",
    batch_size=4,
    epochs=30,
    lr=3e-4,
    val_fraction=0.15,
    num_workers=2,
    seed=42,
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ORIG = 2048

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])

# Locate the dataset wherever Kaggle mounted it (the path differs between competition/dataset inputs).
ANN_PATH = next(Path("/kaggle/input").rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
BASE = ANN_PATH.parent.parent  # .../MAGFiLO_1.0_Kaggle_2026
TRAIN_IMG_DIR = BASE / "train/train_images"
TEST_IMG_DIR = BASE / "test/test_images"
print(DEVICE, BASE)


## 1. Data
Each `image_id` (annotator batch) is a separate training sample, as the organisers suggest. The train/val split is done by **file_name**, so the same Sun image never appears on both sides.

In [ ]:
{
"info": info,
"images": [image],
"annotations": [annotation],
"licenses": [license],
"categories": [category],
}

info{
"year": int,
"version": str,
"description": str,
"contributor": str,
"url": str,
"date_created": datetime,
}

license{
"id": int,
"name": str,
"url": str,
}

categories[
{
   "id": int,
   "name": str,
   "supercategory": str,
}]

annotation{
"id": str,
"image_id": int,
"category_id": int,
"segmentation": [[]],
"area": float,
"spine": [],
"bbox": [x, y, width, height],
"iscrowd": 0 or 1,
}

image{
"id": str,
"width": int,
"height": int,
"file_name": str,
"license": int,
"date_captured": datetime,
}


In [ ]:
with open(ANN_PATH) as f:
    coco = json.load(f)

images_by_id = {im["id"]: im for im in coco["images"]}
anns_by_image_id = defaultdict(list)
for ann in coco["annotations"]:
    anns_by_image_id[ann["image_id"]].append(ann)
ids_by_file = defaultdict(list)
for im in coco["images"]:
    ids_by_file[im["file_name"]].append(im["id"])

file_names = sorted(ids_by_file)
site = [fn[14:16] for fn in file_names]  # GONG instrument code, e.g. 'Bh'
train_files, val_files = train_test_split(file_names, test_size=CFG["val_fraction"],
                                          random_state=CFG["seed"], stratify=site)
train_ids = [i for fn in train_files for i in ids_by_file[fn]]
val_ids = [i for fn in val_files for i in ids_by_file[fn]]
print(f"files train/val: {len(train_files)}/{len(val_files)}  samples train/val: {len(train_ids)}/{len(val_ids)}")

In [ ]:
def instance_map(image_id, size=CFG["img_size"]):
    """(size, size) int32 map: 0 = background, k = k-th filament."""
    im = images_by_id[image_id]
    h, w = im["height"], im["width"]
    inst = np.zeros((h, w), np.int32)
    for k, ann in enumerate(anns_by_image_id[image_id], start=1):
        m = mask_utils.decode(mask_utils.merge(mask_utils.frPyObjects(ann["segmentation"], h, w)))
        inst[m > 0] = k
    return cv2.resize(inst, (size, size), interpolation=cv2.INTER_NEAREST)

def load_image(path, size=CFG["img_size"]):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

# Cache everything in RAM at 1024px (~2 GB) -- decoding 2048 JPEGs every epoch is the bottleneck otherwise.
img_cache = {fn: load_image(TRAIN_IMG_DIR / fn) for fn in tqdm(file_names, desc="images")}
inst_cache = {i: instance_map(i) for i in tqdm(images_by_id, desc="masks")}

In [ ]:
NORM = A.Normalize(mean=(0.449,), std=(0.226,))  # ImageNet grey stats
train_tf = A.Compose([
    A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
    A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.RandomGamma(p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    NORM, ToTensorV2(),
])
val_tf = A.Compose([NORM, ToTensorV2()])

class FilamentDataset(Dataset):
    def __init__(self, image_ids, tf):
        self.ids, self.tf = image_ids, tf
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        iid = self.ids[i]
        img = img_cache[images_by_id[iid]["file_name"]]
        mask = (inst_cache[iid] > 0).astype(np.float32)
        out = self.tf(image=img, mask=mask)
        return out["image"], out["mask"].unsqueeze(0)

train_dl = DataLoader(FilamentDataset(train_ids, train_tf), batch_size=CFG["batch_size"], 
                      shuffle=True, num_workers=CFG["num_workers"], drop_last=True, pin_memory=True)
val_dl   = DataLoader(FilamentDataset(val_ids, val_tf), batch_size=CFG["batch_size"],
                      num_workers=CFG["num_workers"], pin_memory=True)

## 2. Model & training
`smp.Unet` with `in_channels=1` (smp adapts the pretrained first conv). Loss = BCE + Dice from `smp.losses`.

In [ ]:
model = smp.Unet(CFG["encoder"], encoder_weights="imagenet", in_channels=1, classes=1).to(DEVICE)
bce = smp.losses.SoftBCEWithLogitsLoss()
dice = smp.losses.DiceLoss("binary", from_logits=True)
criterion = lambda logits, y: bce(logits, y) + dice(logits, y)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG["lr"],
                                                total_steps=CFG["epochs"] * len(train_dl))
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

In [ ]:
best = float("inf")
for epoch in range(CFG["epochs"]):
    model.train(); tr = 0
    for x, y in tqdm(train_dl, leave=False, desc=f"epoch {epoch}"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = criterion(model(x), y)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tr += loss.item()

    model.eval(); va = 0
    with torch.no_grad(), torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
        for x, y in val_dl:
            va += criterion(model(x.to(DEVICE)), y.to(DEVICE)).item()
    tr, va = tr / len(train_dl), va / len(val_dl)
    if va < best:
        best = va; torch.save(model.state_dict(), "best_unet.pth")
    print(f"epoch {epoch:02d}  train {tr:.4f}  val {va:.4f}{'  *' if va == best else ''}")

## 3. Post-processing tuned on validation PQ
`postprocess` turns a probability map into an instance map:
* **threshold** the probabilities;
* **shape cleanup**: ground truth is smooth hand-drawn polygons without holes, so close small gaps along the outline (`close_r`) and fill holes;
* **merge radius**: label connected components of a *dilated* mask, then keep labels only on the original pixels. Nearby fragments of one filament share an id (fewer one-to-many errors) without fattening the mask;
* **filter**: drop components that are tiny (`min_area`) or that the model is unsure about (mean probability < `min_conf`).

Params are expressed at 1024px and scaled when run at full 2048px resolution.

In [ ]:
def disk(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))

def postprocess(prob, thr, merge_r, min_area, close_r=0, min_conf=0.0, scale=1):
    """Probability map -> instance map (0 = background, k = k-th filament)."""
    mask = (prob > thr).astype(np.uint8)
    if close_r > 0:
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, disk(close_r * scale))
    mask = ndimage.binary_fill_holes(mask).astype(np.uint8)
    grouped = cv2.dilate(mask, disk(merge_r * scale)) if merge_r > 0 else mask
    _, labels = cv2.connectedComponents(grouped, connectivity=8)
    labels *= mask
    drop = [r.label for r in regionprops(labels, intensity_image=prob)
            if r.area < min_area * scale * scale or r.intensity_mean < min_conf]
    labels[np.isin(labels, drop)] = 0
    return labels

@torch.no_grad()
def predict(img_u8):
    """Flip-TTA probability map at CFG['img_size']."""
    x = val_tf(image=img_u8)["image"][None].to(DEVICE)
    flips = [[], [-1], [-2], [-2, -1]]
    with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
        p = sum(torch.sigmoid(model(x.flip(f)).float()).flip(f) for f in flips) / len(flips)
    return p[0, 0]

model.load_state_dict(torch.load("best_unet.pth", map_location=DEVICE)); model.eval()
val_probs = {fn: predict(img_cache[fn]).cpu().numpy() for fn in tqdm(val_files, desc="val predict")}

In [ ]:
def to_panoptic(inst):
    """(H, W) instance map -> (1, H, W, 2) [category, instance] on GPU; filament=1 (thing), background=0 (stuff)."""
    inst = torch.from_numpy(inst.astype(np.int64)).to(DEVICE)
    return torch.stack([(inst > 0).long(), inst], -1)[None]

# PQ's per-pixel torch.unique is ~2 s/image on CPU at 1024px but milliseconds on GPU,
# and the ground truth never changes, so convert it once.
val_gt = {iid: to_panoptic(inst_cache[iid]) for iid in val_ids}

def val_pq(**params):
    """Filament-class (PQ, SQ, RQ) on validation. PQ = SQ x RQ:
    SQ = mean IoU of matched filaments (mask shape), RQ = detection F1 (missed / split / merged / spurious)."""
    pq = PanopticQuality(things={1}, stuffs={0}, return_per_class=True, return_sq_and_rq=True).to(DEVICE)
    for iid in val_ids:
        pred = postprocess(val_probs[images_by_id[iid]["file_name"]], **params)
        pq.update(to_panoptic(pred), val_gt[iid])
    return dict(zip(["PQ", "SQ", "RQ"], pq.compute()[0].tolist()))  # row 0 = filament class

def search(base, grid, desc):
    """Try every combination in `grid` on top of `base`; return the best params and show the top rows."""
    rows = []
    for values in tqdm(list(itertools.product(*grid.values())), desc=desc):
        params = {**base, **dict(zip(grid, values))}
        rows.append((params, val_pq(**params)))
    rows.sort(key=lambda r: -r[1]["PQ"])
    display(pd.DataFrame([{**p, **m} for p, m in rows[:5]]))
    return rows[0]

# Stage 1: threshold / merging / size, without the new shape cleanup and confidence filter.
BASE = dict(thr=0.5, merge_r=0, min_area=50, close_r=0, min_conf=0.0)
BEST, base_scores = search(BASE, dict(thr=[0.3, 0.4, 0.5, 0.6], merge_r=[0, 3, 6, 10, 15],
                                      min_area=[25, 50, 100, 200]), "stage 1")
# Stage 2: outline smoothing + confidence filter on top of the stage-1 best.
BEST, _ = search(BEST, dict(close_r=[0, 2, 4, 6], min_conf=[0.0, 0.5, 0.6, 0.7, 0.8]), "stage 2")
# Stage 3: closing changes how fragments touch, so re-tune threshold and merging.
BEST, scores = search(BEST, dict(thr=[0.3, 0.4, 0.5, 0.6], merge_r=[0, 3, 6, 10, 15]), "stage 3")

print("before cleanup:", {k: round(v, 4) for k, v in base_scores.items()})
print("after  cleanup:", {k: round(v, 4) for k, v in scores.items()})
print("best params:", BEST)

In [ ]:
import matplotlib.pyplot as plt
fn = val_files[0]; iid = ids_by_file[fn][0]
pred = postprocess(val_probs[fn], **BEST)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
for a, (im, t) in zip(ax, [(img_cache[fn], fn), (inst_cache[iid], "ground truth"), (pred, "prediction")]):
    a.imshow(im, cmap="gray" if t == fn else "nipy_spectral", interpolation="nearest"); a.set_title(t); a.axis("off")
plt.show()

## 4. Test inference → `submission.csv`
Probabilities are upsampled to 2048 before thresholding, so masks are at full resolution.

In [ ]:
def rle_counts(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode()

rows = []
for path in tqdm(sorted(TEST_IMG_DIR.glob("*.jpeg")), desc="test"):
    prob = predict(load_image(path))
    prob = torch.nn.functional.interpolate(prob[None, None], size=(ORIG, ORIG), mode="bilinear")[0, 0].cpu().numpy()
    labels = postprocess(prob, **BEST, scale=ORIG // CFG["img_size"])
    ids = [k for k in np.unique(labels) if k != 0]
    for n, k in enumerate(ids, start=1):
        rows.append({"filament_id": f"{path.stem}_{n}", "segmentation_rle": rle_counts(labels == k)})
    if not ids:  # every test image needs at least one row
        rows.append({"filament_id": f"{path.stem}_1", "segmentation_rle": rle_counts(np.zeros((ORIG, ORIG)))})

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
print(sub.shape, sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
sub.head()